# 🚀 Research-Ready Activation Pruning Pipeline (V4 - Optimized)

### ✨ Key Fixes in V4:
1.  **CIFAR-Optimized VGG16**: Classifier shrunk from 100M params to ~1M, enabling fast convergence on 32x32 images.
2.  **Convergence Boost**: Switched to Adam(3e-4) to instantly break the 10% accuracy plateau.
3.  **End-to-End Dependency Fix**: Robust mapping for Conv -> Linear transitions.

In [5]:
!pip -q install thop seaborn tqdm matplotlib

import torch
import torch.nn as nn
import torch.optim as optim
import copy, os, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from abc import ABC, abstractmethod
from typing import Dict, List
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm
from thop import profile, clever_format
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# =========================================
# 1. RESEARCHER API (The Pruning Math)
# =========================================
class BaseCriterion(ABC):
    @abstractmethod
    def compute_scores(self, activations: torch.Tensor) -> torch.Tensor: pass

class MeanAbsCriterion(BaseCriterion):
    def compute_scores(self, activations): return activations.abs().mean(dim=(0, 2, 3))

# =========================================
# 2. HEAVY LIFTING (Surgery Engine)
# =========================================
class PruningEngine:
    def __init__(self, model):
        self.model = model
        self.dependencies = self._trace_dependencies()

    def _trace_dependencies(self):
        mapping = {}
        mods = list(self.model.named_modules())
        for i, (name, m) in enumerate(mods):
            if isinstance(m, nn.Conv2d):
                siblings = []
                if i+1 < len(mods) and isinstance(mods[i+1][1], nn.BatchNorm2d): siblings.append(mods[i+1][0])
                next_layers = []
                for j in range(i+1, len(mods)):
                    if isinstance(mods[j][1], (nn.Conv2d, nn.Linear)): 
                        next_layers.append(mods[j][0])
                        break
                mapping[name] = {"siblings": siblings, "next_layers": next_layers, "out_channels": m.out_channels}
        return mapping

    def apply_masks(self, model, masks: Dict[str, torch.Tensor]):
        new_model = copy.deepcopy(model)
        for layer_name, mask in masks.items():
            idx = torch.where(mask)[0]
            orig_out = self.dependencies[layer_name]["out_channels"]
            self._shrink(new_model, layer_name, idx, dim=0)
            for sib in self.dependencies[layer_name]["siblings"]: self._shrink(new_model, sib, idx, dim=0)
            for nxt in self.dependencies[layer_name]["next_layers"]: self._shrink(new_model, nxt, idx, dim=1, source_channels=orig_out)
        return new_model

    def _shrink(self, model, name, idx, dim, source_channels=None):
        parts = name.split('.'); curr = model
        for p in parts[:-1]: curr = getattr(curr, p)
        m = getattr(curr, parts[-1])
        if isinstance(m, nn.Conv2d):
            if dim == 0:
                m.weight = nn.Parameter(m.weight.data[idx]); m.out_channels = len(idx)
                if m.bias is not None: m.bias = nn.Parameter(m.bias.data[idx])
            else:
                m.weight = nn.Parameter(m.weight.data[:, idx]); m.in_channels = len(idx)
        elif isinstance(m, nn.BatchNorm2d):
            m.weight = nn.Parameter(m.weight.data[idx]); m.bias = nn.Parameter(m.bias.data[idx])
            m.running_mean, m.running_var, m.num_features = m.running_mean[idx], m.running_var[idx], len(idx)
        elif isinstance(m, nn.Linear):
            if dim == 1:
                scale = m.in_features // source_channels
                new_idx = torch.cat([idx * scale + s for s in range(scale)]).sort()[0]
                m.weight = nn.Parameter(m.weight.data[:, new_idx]); m.in_features = len(new_idx)

# =========================================
# 3. PIPELINE ORCHESTRATOR
# =========================================
class PruningPipeline:
    def __init__(self, config):
        self.config = config
        self.path = f"experiments/{config['name']}_baseline.pth"
        os.makedirs("experiments", exist_ok=True)
        
    def get_cifar_vgg(self):
        # Optimized VGG for CIFAR-10
        m = models.vgg16_bn(num_classes=10)
        m.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        m.classifier = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(True), nn.Dropout(),
            nn.Linear(512, 10)
        )
        return m.to(device)

    def train_baseline(self, model, loader):
        if not self.config['force_retrain'] and os.path.exists(self.path):
            model.load_state_dict(torch.load(self.path, map_location=device)['state_dict'])
            return
        print(f"🚀 Training Baseline...")
        opt = optim.Adam(model.parameters(), lr=3e-4)
        crit = nn.CrossEntropyLoss()
        for e in range(self.config['epochs']):
            model.train(); total_loss = 0
            for x, y in tqdm(loader, desc=f"Epoch {e+1}", leave=False):
                x, y = x.to(device), y.to(device)
                opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
                total_loss += loss.item()
            acc = self.evaluate(model, loader)
            print(f"Epoch {e+1}: Loss {total_loss/len(loader):.4f} | Acc {acc:.2f}%")
            if acc >= 75.0: break
        torch.save({'state_dict': model.state_dict()}, self.path)

    def evaluate(self, model, loader):
        model.eval(); correct, total = 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
        return 100 * correct / total

    def run_pruning(self, model, loader):
        engine = PruningEngine(model); scores = {}
        model.eval(); x, _ = next(iter(loader)); acts = {}
        def hook(n): return lambda m, i, o: acts.update({n: o})
        hooks = [m.register_forward_hook(hook(n)) for n, m in model.named_modules() if isinstance(m, nn.Conv2d)]
        model(x.to(device))
        for h in hooks: h.remove()
        for n, a in acts.items(): scores[n] = self.config['criterion_math'].compute_scores(a)
        masks = {l: (torch.topk(s, int(self.config['ratio']*s.numel()), largest=False)[1]) for l, s in scores.items()}
        final_masks = {l: torch.ones(s.numel(), dtype=torch.bool).to(device) for l, s in scores.items()}
        for l, p_idx in masks.items(): final_masks[l][p_idx] = False
        pruned = engine.apply_masks(model, final_masks)
        self.analyze(model, pruned)
        return pruned

    def analyze(self, base, pruned):
        def stats(m):
            f, p = profile(copy.deepcopy(m).cpu(), inputs=(torch.randn(1, 3, 32, 32),), verbose=False)
            return clever_format([f, p], "%.2f")
        f_b, p_b = stats(base); f_p, p_p = stats(pruned)
        print(f"\n📊 Baseline: {p_b} Params | {f_b} FLOPs\n📊 Pruned:   {p_p} Params | {f_p} FLOPs")

# =========================================
# 4. EXECUTION
# =========================================
config = {'name': 'vgg_optimized', 'epochs': 20, 'ratio': 0.5, 'force_retrain': False, 'criterion_math': MeanAbsCriterion()}
tf = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.49, 0.48, 0.44), (0.20, 0.19, 0.20))])
train_loader = DataLoader(datasets.CIFAR10('./data', train=True, download=True, transform=tf), batch_size=128, shuffle=True)

pipeline = PruningPipeline(config)
model = pipeline.get_cifar_vgg()
pipeline.train_baseline(model, train_loader)
pruned_model = pipeline.run_pruning(model, train_loader)

print(f"\n✅ Done! Orig Acc: {pipeline.evaluate(model, train_loader):.2f}% -> Pruned: {pipeline.evaluate(pruned_model, train_loader):.2f}%")

Using device: cuda
🚀 Training Baseline...


Epoch 1: Loss 1.2931 | Acc 63.51%


Epoch 2: Loss 0.8366 | Acc 76.61%

📊 Baseline: 14.99M Params | 314.57M FLOPs
📊 Pruned:   3.82M Params | 79.43M FLOPs

✅ Done! Orig Acc: 76.73% -> Pruned: 10.00%
